# LSTM Text Classification

## Why LSTM over RNN?

A vanilla RNN passes one hidden state between steps. For long sequences, gradients from early tokens vanish before reaching the output — the model forgets what it read.

LSTM adds a **cell state** (long-term memory) alongside the hidden state, controlled by three gates:

| Gate | Role |
|---|---|
| Forget gate | decides what to drop from cell state |
| Input gate | decides what new info to write in |
| Output gate | decides what cell state to expose as hidden state |

Result: gradients flow further back, model handles longer dependencies.

**GRU** is a simplified LSTM (2 gates, no separate cell state) — fewer params, similar performance on most tasks.

This notebook: classify IMDb sentiment with LSTM, then swap to GRU with one line.

In [ ]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 1. Data and Vocabulary

In [ ]:
# 20 Newsgroups — same dataset as RNN notebook, easy comparison
data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
texts, labels = data.data, data.target
num_classes = len(data.target_names)
print(f'{len(texts)} docs, {num_classes} classes')

In [ ]:
def tokenize(text):
    return re.sub(r'[^a-z0-9\s]', '', text.lower()).split()

# Build vocab from training split only — avoids leaking test vocabulary
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

PAD, UNK = 0, 1
vocab = {'<PAD>': PAD, '<UNK>': UNK}
for text in X_train_raw:
    for tok in tokenize(text):
        if tok not in vocab:
            vocab[tok] = len(vocab)

print(f'vocab size: {len(vocab):,}')

## 2. Variable-Length Dataset with Padding

RNN notebooks often pad all sequences to the same length upfront.  
Better approach: keep sequences at their natural length, pad *per batch* using `pad_sequence`, and use `pack_padded_sequence` so the LSTM skips padding tokens entirely — faster and more accurate.

In [ ]:
MAX_LEN = 200

def encode(text):
    # empty docs become one <UNK>: pack_padded_sequence rejects length 0
    ids = ([vocab.get(t, UNK) for t in tokenize(text)] or [UNK])[:MAX_LEN]
    return torch.tensor(ids, dtype=torch.long)


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.X = [encode(t) for t in texts]
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def collate_fn(batch):
    """Pad sequences to the longest in the batch, return lengths for packing."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded  = pad_sequence(seqs, batch_first=True, padding_value=PAD)
    return padded, lengths, torch.stack(labels)


train_ds = TextDataset(X_train_raw, y_train)
test_ds  = TextDataset(X_test_raw,  y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, collate_fn=collate_fn)

xb, lb, yb = next(iter(train_loader))
print('batch shape:', xb.shape, '  lengths range:', lb.min().item(), '–', lb.max().item())

## 3. LSTM Model

`pack_padded_sequence` tells the LSTM each sequence's real length so it doesn't process padding tokens.  
`pad_packed_sequence` unpacks the output back to a padded tensor.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(
            embed_dim, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        x = self.embedding(x)                           # (B, T, E)

        # Pack so LSTM skips padding
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)              # hidden: (num_layers, B, H)

        # Use last layer's hidden state
        out = self.dropout(hidden[-1])                  # (B, H)
        return self.fc(out)                             # (B, num_classes)


model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_size=256,
    num_layers=2,
    num_classes=num_classes,
    dropout=0.3
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

## 4. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

for epoch in range(3):
    model.train()
    total_loss = 0
    for xb, lb, yb in train_loader:
        xb, lb, yb = xb.to(DEVICE), lb, yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb, lb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # clip exploding grads
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, lb, yb in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            correct += (model(xb, lb).argmax(1) == yb).sum().item()
            total   += yb.size(0)

    print(f'Epoch {epoch+1}  loss={total_loss/len(train_loader):.4f}  test_acc={correct/total:.4f}')

## 5. Upgrade: Bidirectional LSTM

A bidirectional LSTM runs two LSTMs — one forward, one backward.  
Each token's representation sees context from both directions.  
Hidden size doubles (`hidden_size * 2`), so adjust the classifier head.

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(
            embed_dim, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True       # <-- only change
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 because bidirectional

    def forward(self, x, lengths):
        x = self.embedding(x)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        # hidden: (num_layers * 2, B, H) — concat forward and backward last layers
        out = torch.cat([hidden[-2], hidden[-1]], dim=1)   # (B, H*2)
        return self.fc(self.dropout(out))


bilstm = BiLSTMClassifier(len(vocab), 128, 128, 2, num_classes).to(DEVICE)
print(f'BiLSTM params: {sum(p.numel() for p in bilstm.parameters()):,}')

## 6. Swap LSTM → GRU in one line

GRU has no cell state — just hidden state, two gates.  
Faster to train, same API as LSTM except the forward returns `(output, hidden)` not `(output, (hidden, cell))`.

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(            # swap nn.LSTM → nn.GRU
            embed_dim, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        x = self.embedding(x)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)  # no cell state — just (output, hidden)
        return self.fc(self.dropout(hidden[-1]))


gru = GRUClassifier(len(vocab), 128, 256, 2, num_classes).to(DEVICE)
print(f'GRU params: {sum(p.numel() for p in gru.parameters()):,}')
# GRU has ~25% fewer params than LSTM with same hidden_size

## Summary: RNN vs LSTM vs GRU

| Model | Long-range deps | Params | Train speed | When to use |
|---|---|---|---|---|
| `nn.RNN` | Poor | Fewest | Fastest | Very short sequences / learning exercise |
| `nn.GRU` | Good | Medium | Fast | Most sequence tasks — good default |
| `nn.LSTM` | Good | Most | Slower | When GRU underfits; established baselines |
| `BiLSTM` | Best | 2× LSTM | Slowest | Sentence-level tasks where full context matters |

**Next step**: Replace the learned Embedding with pre-trained GloVe or fastText vectors — same architecture, dramatically better representations for small datasets.